🛡️ 02 — Async Examples: 3 Real-Feel Mini-Projects

**Topic 12.02 | Async/Parallel Python — Part 2 of 4 (examples notebook)**

🌍 **Hook:** Theory is done — now 3 things you'll want in your daily apps:
1. 📊 **API-DASH** — 3 different "APIs" all fetched at the same time (gather combo-report!)
2. 📥 **PROGRESS-DOWNLOADS** — LIVE progress of downloads (async as_completed!)
3. 🛡️ **TIMEOUT-SHIELD** — on a slow API: "10s wait? CANCEL!" (wait_for guard)

All three are seed-fixed — identical results on your machine too! 🎯

## 🐣 Setup + universal run() (the key from the previous notebook!)

### Sets
Unordered collections of unique elements. Useful for membership tests and set operations (union/intersection).

In [1]:
import asyncio, random, time
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

def run(coro):
    """notebook + .py dono jagah chalao"""
    return asyncio.run(coro)

print("🛡️ examples ready — 3 mini-projects line me!")
# Expected OUTPUT:
# 🛡️ examples ready — 3 mini-projects line me!

🛡️ examples ready — 3 mini-projects line me!


## 📊 Mini-1 — API-DASH: 3 APIs, One Digest

📖 **Scene:** A dashboard needs: student-count, weather-avg, zomato-rating —
THREE separate services (each takes 0.x s). Adding them synchronously = 0.9s; gather = just the slowest one.

In [2]:
random.seed(21)

async def api_students():
    await asyncio.sleep(0.3)
    return {"service": "students", "count": 50, "top_city": "Delhi"}

async def api_weather():
    await asyncio.sleep(0.2)
    return {"service": "weather", "avg_temp": 24.9, "hottest": "2024-08-15"}

async def api_zomato():
    await asyncio.sleep(0.35)
    return {"service": "zomato", "restaurants": 100, "avg_rating": 3.94}

async def digest():
    t0 = time.perf_counter()
    s, w, z = await asyncio.gather(api_students(), api_weather(), api_zomato())
    el = time.perf_counter() - t0
    print(f"⏱️ 3 APIs in {el:.2f}s (sync-zod = 0.85s, gather = slowest-ek ≈ 0.35s!)")
    print("📊 DASH-DIGEST:")
    print(f"   👨‍🎓 students: {s['count']} (top city {s['top_city']})")
    print(f"   🌡️ weather : avg {w['avg_temp']}°C (hottest {w['hottest']})")
    print(f"   🍽️ zomato  : {z['restaurants']} spots, avg-rating {z['avg_rating']}/5")

run(digest())
# Expected OUTPUT (seed-independent — waits fixed isi code me):
# ⏱️ 3 APIs in ~0.35s (sync-zod = 0.85s, gather ≈ 0.35s!)
# 📊 DASH-DIGEST:
#    👨‍🎓 students: 50 (top city Delhi)
#    🌡️ weather : avg 24.9°C (hottest 2024-08-15)
#    🍽️ zomato  : 100 spots, avg-rating 3.94/5

⏱️ 3 APIs in 0.35s (sync-zod = 0.85s, gather = slowest-ek ≈ 0.35s!)
📊 DASH-DIGEST:
   👨‍🎓 students: 50 (top city Delhi)
   🌡️ weather : avg 24.9°C (hottest 2024-08-15)
   🍽️ zomato  : 100 spots, avg-rating 3.94/5


## 📥 Mini-2 — Progress Downloads (as_completed LIVE!)

📖 **Definition:** `asyncio.as_completed(tasks)` = yields the coroutines in their
FINISHING order — as soon as a download completes, its progress shows right away!

🔍 **WHY is the async version so exciting?** The threading version of as_completed works on
futures; here you get live progress in PURE single-thread async too — just one hand! 🦸

In [3]:
async def download(fid, secs):
    await asyncio.sleep(secs)
    return f"file-{fid} ({secs*1000:.0f}KB)"

async def progress_dl():
    plan = [(0, 0.3), (1, 0.1), (2, 0.2), (3, 0.15)]      # (id, simulated-time)
    tasks = [download(i, s) for i, s in plan]
    t0 = time.perf_counter()
    done = 0
    for fut in asyncio.as_completed(tasks):
        result = await fut
        done += 1
        bar = "▓" * done + "░" * (len(plan) - done)
        print(f"   📥 {result} aaya! progress [{bar}] {done}/{len(plan)} "
              f"({time.perf_counter()-t0:.2f}s)")

run(progress_dl())
# Expected OUTPUT (finishing-order: 0.1→0.15→0.2→0.3 — chhoti pehle!):
#    📥 file-1 (100KB) aaya! progress [▓░░░] 1/4 (0.10s)
#    📥 file-3 (150KB) aaya! progress [▓▓░░] 2/4 (0.15s)
#    📥 file-2 (200KB) aaya! progress [▓▓▓░] 3/4 (0.20s)
#    📥 file-0 (300KB) aaya! progress [▓▓▓▓] 4/4 (0.30s)

   📥 file-1 (100KB) aaya! progress [▓░░░] 1/4 (0.10s)
   📥 file-3 (150KB) aaya! progress [▓▓░░] 2/4 (0.15s)
   📥 file-2 (200KB) aaya! progress [▓▓▓░] 3/4 (0.20s)
   📥 file-0 (300KB) aaya! progress [▓▓▓▓] 4/4 (0.30s)


## 🛡️ Mini-3 — TIMEOUT-SHIELD: Give the Slow API a Red-Card!

📖 **Definition:** `await asyncio.wait_for(coro, timeout=t)` = give the coroutine a
t-second deadline — if it crosses it, you get a `TimeoutError` (cancelled!). Even if the
API hangs, YOU don't — just show a fallback.

⚠️ **Mistake:** assuming a timeout "will work" without testing it against a real server —
always drill with a fake-slow API first (like below!).

In [4]:
async def lazy_api(delay):
    await asyncio.sleep(delay)
    return "real-data-mil-gayi!"

async def shielded_call():
    # Case-A: fast API (0.1s < 0.25s timeout) → success
    try:
        res = await asyncio.wait_for(lazy_api(0.1), timeout=0.25)
        print(f"   ✅ fast-API: {res}")
    except TimeoutError:
        print("   (nahi aana chahiye tha yahan!)")

    # Case-B: lazy API (0.5s > 0.25s timeout) → shield kicks in 🛡️
    try:
        res = await asyncio.wait_for(lazy_api(0.5), timeout=0.25)
        print(f"   {res}")
    except TimeoutError:
        print("   🛡️ SHIELD ON: lazy-API 0.25s me nahi aayi → CANCEL + fallback!")
        res = "CACHED-DATA (fallback 🗄️)"
        print(f"   user ko dikha: {res}")

run(shielded_call())
# Expected OUTPUT:
#    ✅ fast-API: real-data-mil-gayi!
#    🛡️ SHIELD ON: lazy-API 0.25s me nahi aayi → CANCEL + fallback!
#    user ko dikha: CACHED-DATA (fallback 🗄️)

   ✅ fast-API: real-data-mil-gayi!
   🛡️ SHIELD ON: lazy-API 0.25s me nahi aayi → CANCEL + fallback!
   user ko dikha: CACHED-DATA (fallback 🗄️)


---
## ⚠️ 3 Combo-Specific Mistakes
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Assuming the results order (as_completed ≠ map!) | ask for the finishing order only |
| 2 | Trusting every API without wait_for | always use the shield (production!) |
| 3 | gather without try-any-Exception (everything fails) | gather(return_exceptions=True) or per-task try |

## 📌 Recap — 30 Seconds
- `gather` = combo-digest (time ≈ the slowest one!) 📊
- `as_completed` = live-progress (finishing order!) 📥
- `wait_for` = timeout-shield (red-card for slow APIs 🛡️)

**Next:** `task.ipynb` — 7 async-orders on the Order-Board + the BOSS! 🎫